# 🚢 Titanic Feature Engineering
## AI Assignment 2 — Full Pipeline
---
This notebook walks through all three assignment parts:

| Part | Topic | Marks |
|------|-------|-------|
| 1 | Data Cleaning | 10 |
| 2 | Feature Engineering | 30 |
| 3 | Feature Selection | 10 |

---


## 📦 Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
import os

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110
os.makedirs("figures", exist_ok=True)

print("All imports OK ✓")


---
# Part 1 — Data Cleaning
## 1.1 Load & Inspect

In [ ]:
df = pd.read_csv("../data/train.csv")
print(f"Shape: {df.shape}")
df.head()


In [ ]:
df.info()


In [ ]:
df.describe()


## 1.2 Missing Value Analysis

In [ ]:
missing = df.isnull().sum()
pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({
    "Missing Count": missing,
    "Missing %": pct
}).sort_values("Missing %", ascending=False)
missing_report[missing_report["Missing Count"] > 0]


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]
missing_pct.plot(kind="bar", ax=ax, color="salmon", edgecolor="black")
ax.set_ylabel("Missing %")
ax.set_title("Missing Values by Column")
ax.axhline(50, color="red", linestyle="--", label="50% threshold")
ax.legend()
plt.tight_layout()
plt.savefig("figures/missing_values.png")
plt.show()


### Decision Log — Missing Values

| Column | Missing % | Strategy | Reason |
|--------|-----------|----------|--------|
| **Cabin** | ~77% | Drop column | Too many missing; Deck letter extracted first |
| **Age** | ~20% | Median imputation + `AgeIsMissing` flag | Median is robust to skew; flag preserves information |
| **Embarked** | <1% | Mode imputation | Only 2 rows; mode is safe |
| **Fare** | <1% | Median imputation | Only 1 row (test set) |


In [ ]:
# Age: median + indicator flag
df["AgeIsMissing"] = df["Age"].isnull().astype(int)
df["Age"] = df["Age"].fillna(df["Age"].median())

# Embarked: mode
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

# Fare: median
df["Fare"] = df["Fare"].fillna(df["Fare"].median())

print("Remaining missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("All critical columns clean ✓" if df[["Age","Embarked","Fare"]].isnull().sum().sum() == 0 else "Still missing!")


## 1.3 Outlier Detection & Handling

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for i, col in enumerate(["Age", "Fare"]):
    # Before
    axes[0][i].set_title(f"{col} — Before Capping")
    sns.boxplot(x=df[col], ax=axes[0][i], color="lightblue")

# Cap using IQR × 3
for col in ["Fare", "Age"]:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    upper = Q3 + 3 * IQR
    lower = max(0, Q1 - 3 * IQR)
    n = ((df[col] > upper) | (df[col] < lower)).sum()
    df[col] = df[col].clip(lower=lower, upper=upper)
    print(f"{col}: capped {n} outliers → [{lower:.1f}, {upper:.1f}]")

for i, col in enumerate(["Age", "Fare"]):
    axes[1][i].set_title(f"{col} — After Capping")
    sns.boxplot(x=df[col], ax=axes[1][i], color="lightgreen")

plt.tight_layout()
plt.savefig("figures/outliers.png")
plt.show()


### Outlier Decision
- **Fare**: upper fence = Q3 + 3×IQR (use 3× instead of 1.5× because high fare may signal survival).
- **Age**: same conservative IQR cap. No negative ages possible, so lower bound = max(0, ...).


## 1.4 Data Consistency

In [ ]:
print("Sex unique values:", df["Sex"].unique())
df["Sex"] = df["Sex"].str.lower().str.strip()
df["Embarked"] = df["Embarked"].str.strip()

before = len(df)
df = df.drop_duplicates()
print(f"Duplicates removed: {before - len(df)}")
print("Consistency checks passed ✓")


## 1.5 Save Cleaned Dataset

In [ ]:
df.to_csv("../data/train_cleaned.csv", index=False)
print(f"Saved train_cleaned.csv — shape: {df.shape}")
df.head()


---
# Part 2 — Feature Engineering
## 2.1 Family Features

In [ ]:
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x="FamilySize", hue="Survived", ax=axes[0])
axes[0].set_title("Survival by Family Size")
sns.countplot(data=df, x="IsAlone", hue="Survived", ax=axes[1])
axes[1].set_xticklabels(["Not Alone", "Alone"])
axes[1].set_title("Survival: Alone vs With Family")
plt.tight_layout()
plt.savefig("figures/family_features.png")
plt.show()
print("Survival rate alone:", df[df["IsAlone"]==1]["Survived"].mean().round(3))
print("Survival rate with family:", df[df["IsAlone"]==0]["Survived"].mean().round(3))


## 2.2 Title Extraction from Name

In [ ]:
df["Title"] = df["Name"].str.extract(r",\s*([^.]+)\.")
print("Raw titles:\n", df["Title"].value_counts().to_string())


In [ ]:
title_map = {
    "Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs",
    "Lady": "Royalty", "Countess": "Royalty", "Capt": "Officer",
    "Col": "Officer", "Don": "Royalty", "Dr": "Officer",
    "Major": "Officer", "Rev": "Officer", "Sir": "Royalty",
    "Jonkheer": "Royalty", "Dona": "Royalty",
}
df["Title"] = df["Title"].replace(title_map)
common = {"Mr", "Mrs", "Miss", "Master", "Officer", "Royalty"}
df["Title"] = df["Title"].apply(lambda t: t if t in common else "Other")

fig, ax = plt.subplots(figsize=(9, 4))
sns.countplot(data=df, x="Title", hue="Survived", order=df["Title"].value_counts().index, ax=ax)
ax.set_title("Survival by Title")
plt.tight_layout()
plt.savefig("figures/title_survival.png")
plt.show()


## 2.3 Deck Extraction from Cabin

In [ ]:
df["Deck"] = df["Cabin"].apply(
    lambda c: re.findall(r"[A-Z]", str(c))[0] if pd.notna(c) and str(c) != "nan" else "U"
)
print("Deck distribution:")
print(df["Deck"].value_counts())

fig, ax = plt.subplots(figsize=(10, 4))
deck_survival = df.groupby("Deck")["Survived"].mean().sort_values(ascending=False)
deck_survival.plot(kind="bar", ax=ax, color="teal", edgecolor="black")
ax.set_ylabel("Survival Rate")
ax.set_title("Survival Rate by Deck")
plt.tight_layout()
plt.savefig("figures/deck_survival.png")
plt.show()


## 2.4 Age Groups

In [ ]:
bins = [0, 12, 17, 60, 120]
labels = ["Child", "Teen", "Adult", "Senior"]
df["AgeGroup"] = pd.cut(df["Age"], bins=bins, labels=labels, right=True)

fig, ax = plt.subplots(figsize=(8, 4))
sns.countplot(data=df, x="AgeGroup", hue="Survived",
              order=["Child", "Teen", "Adult", "Senior"], ax=ax)
ax.set_title("Survival by Age Group")
plt.tight_layout()
plt.savefig("figures/age_group_survival.png")
plt.show()


## 2.5 Fare Per Person

In [ ]:
df["FarePerPerson"] = df["Fare"] / df["FamilySize"]
print(df["FarePerPerson"].describe())


## 2.6 Interaction Features

In [ ]:
df["Pclass_x_Fare"] = df["Pclass"] * df["Fare"]
df["Age_x_Pclass"]  = df["Age"]   * df["Pclass"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col in zip(axes, ["Pclass_x_Fare", "Age_x_Pclass"]):
    sns.boxplot(data=df, x="Survived", y=col, ax=ax, palette="Set2")
    ax.set_title(f"{col} vs Survived")
plt.tight_layout()
plt.savefig("figures/interaction_features.png")
plt.show()


## 2.7 Log Transformations (skewed features)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for i, col in enumerate(["Fare", "FarePerPerson"]):
    sns.histplot(df[col], ax=axes[0][i], kde=True, color="salmon")
    axes[0][i].set_title(f"{col} — Original")

for col in ["Fare", "FarePerPerson"]:
    df[f"Log_{col}"] = np.log1p(df[col])

for i, col in enumerate(["Log_Fare", "Log_FarePerPerson"]):
    sns.histplot(df[col], ax=axes[1][i], kde=True, color="steelblue")
    axes[1][i].set_title(f"{col} — After log1p")

plt.tight_layout()
plt.savefig("figures/log_transforms.png")
plt.show()


## 2.8 Categorical Encoding (One-Hot)

In [ ]:
ohe_cols = ["Sex", "Embarked", "Title", "Deck", "AgeGroup"]
df = pd.get_dummies(df, columns=ohe_cols, drop_first=False)
bool_cols = df.select_dtypes(include="bool").columns
df[bool_cols] = df[bool_cols].astype(int)

# Drop raw/ID columns
drop_cols = ["Name", "Ticket", "Cabin", "PassengerId"]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

print(f"Shape after encoding: {df.shape}")
df.head()


## 2.9 Feature Scaling (StandardScaler)

In [ ]:
scale_cols = ["Age", "Fare", "FamilySize", "FarePerPerson",
              "Log_Fare", "Log_FarePerPerson"]
scale_cols = [c for c in scale_cols if c in df.columns]

scaler = StandardScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])

print("Scaled columns:", scale_cols)
df[scale_cols].describe().round(2)


In [ ]:
df.to_csv("../data/train_engineered.csv", index=False)
print(f"Saved train_engineered.csv — shape: {df.shape}")


---
# Part 3 — Feature Selection
## 3.1 Correlation Analysis

In [ ]:
TARGET = "Survived"
num_df = df.select_dtypes(include=[np.number])

plt.figure(figsize=(18, 14))
corr = num_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap="coolwarm", center=0,
            linewidths=0.3, annot=False, square=True)
plt.title("Correlation Heatmap (lower triangle)")
plt.tight_layout()
plt.savefig("figures/correlation_heatmap.png")
plt.show()


In [ ]:
# Drop features with pairwise correlation > 0.90
upper = corr.abs().where(np.triu(np.ones(corr.shape), k=1).astype(bool))
high_corr = [col for col in upper.columns if any(upper[col] > 0.90)]
print("High-correlation features to drop:", high_corr)
df = df.drop(columns=high_corr)
print(f"Shape after dropping: {df.shape}")


## 3.2 Random Forest Feature Importance

In [ ]:
X = df.drop(columns=[TARGET]).select_dtypes(include=[np.number])
y = df[TARGET]

rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(X, y)

importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
importances.head(20).sort_values().plot(kind="barh", ax=ax, color="steelblue", edgecolor="black")
ax.set_xlabel("Importance Score")
ax.set_title("Top 20 Feature Importances (Random Forest)")
plt.tight_layout()
plt.savefig("figures/feature_importance.png")
plt.show()


In [ ]:
threshold = importances.mean()
selected_rf = importances[importances >= threshold].index.tolist()
print(f"Mean importance threshold: {threshold:.4f}")
print(f"RF-selected features ({len(selected_rf)}): {selected_rf}")


## 3.3 Recursive Feature Elimination (RFE) — Extra Credit

In [ ]:
estimator = LogisticRegression(max_iter=1000, random_state=42)
rfe = RFE(estimator, n_features_to_select=15, step=1)
rfe.fit(X, y)

selected_rfe = X.columns[rfe.support_].tolist()
rfe_ranking = pd.Series(rfe.ranking_, index=X.columns).sort_values()
print("RFE Ranking (top 20):")
print(rfe_ranking.head(20).to_string())
print(f"\nRFE selected ({len(selected_rfe)}): {selected_rfe}")


## 3.4 Final Feature Set

In [ ]:
final_features = list(set(selected_rf) | set(selected_rfe) | {TARGET})
final_features = [f for f in final_features if f in df.columns]

df_selected = df[final_features]
df_selected.to_csv("../data/train_selected.csv", index=False)
print(f"Final dataset: {df_selected.shape[1]-1} features + target")
print("Features saved → ../data/train_selected.csv")
df_selected.head()


## 3.5 Feature Justification

| Feature | Kept? | Reason |
|---------|-------|--------|
| `Sex_*` | ✅ | Strongest single predictor (women/children first) |
| `Pclass` | ✅ | Socioeconomic proxy, strong correlation with survival |
| `Title_*` | ✅ | Captures age/gender/class simultaneously |
| `Age` / `AgeGroup_*` | ✅ | Children survived at higher rates |
| `Log_Fare` / `FarePerPerson` | ✅ | Socioeconomic signal, better distributed after log |
| `FamilySize` / `IsAlone` | ✅ | Medium-sized families survived better |
| `Embarked_*` | ✅ | Correlated with class/fare, mild predictive value |
| `Deck_*` | ✅ | Proxy for cabin location — distance to lifeboats |
| `AgeIsMissing` | ✅ | Missingness itself may carry information |
| `SibSp`, `Parch` | ❌ | Subsumed by `FamilySize` |
| `Ticket` | ❌ | Free text, no direct signal |
| `Name` | ❌ | Captured via `Title` |
| `Cabin` (raw) | ❌ | Captured via `Deck`; 77% missing |


---
## ✅ Summary

All three assignment parts are complete:
- **Part 1**: Cleaned data saved to `data/train_cleaned.csv`
- **Part 2**: Engineered features saved to `data/train_engineered.csv`
- **Part 3**: Selected features saved to `data/train_selected.csv`

Visualisations are in the `figures/` directory.